# Module 6 EDA practice: 8 open questions

**Video Module 6 practice** | ShopKart project

**Goal:** Practise EDA on shopkart_clean.csv. Answer each question with code, then write 1-2 sentences of finding in plain English.

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `shopkart_clean.csv`.
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

In [1]:
import pandas as pd
df = pd.read_csv("shopkart_clean.csv", parse_dates=["order_date"])
df["is_returned"] = (df["returned"] == "Y").astype(int)
df.shape

(5000, 22)

## Q1. Which product brought the most revenue in 2025? List the top 5 products by revenue.

In [2]:
top = df.groupby("product_name")["revenue"].sum().sort_values(ascending=False).round(0)
top.head(5)

product_name
Tablet                 2522291.0
Smartphone             1968580.0
Air Fryer               847788.0
Smartwatch              585126.0
Mechanical Keyboard     491440.0
Name: revenue, dtype: float64

**Finding:** Tablet (₹25.22 lakh) and Smartphone (₹19.69 lakh) are far ahead of everything else. Air Fryer (₹8.48 lakh) is third. A few expensive Electronics items drive a large share of revenue.

## Q2. Which category has the best profit margin (profit / revenue)?

In [3]:
cat = df.groupby("category")[["revenue", "profit"]].sum()
cat["margin_%"] = (cat["profit"] / cat["revenue"] * 100).round(1)
cat.sort_values("margin_%", ascending=False)

,revenue,profit,margin_%
category,,,
Beauty,877640.35,375400.35,42.8
Fashion,3121345.60,1130545.60,36.2
Home,2655047.30,689997.30,26.0
Books,719598.40,136338.40,18.9
Electronics,7331018.05,808468.05,11.0


**Finding:** Beauty has the best margin (42.8%), then Fashion (36.2%). Electronics brings half of all revenue but keeps only 11.0% as profit. Fashion makes the most total profit (₹11.31 lakh).

## Q3. How does delivery time relate to rating? Show average rating per delivery_days and the correlation.

In [4]:
print(df.groupby("delivery_days")["rating"].mean().round(2))
print("Correlation:", round(df["delivery_days"].corr(df["rating"]), 2))

delivery_days
2     4.20
3     4.22
4     4.05
5     3.81
6     3.63
7     3.39
8     3.08
9     2.92
10    3.17
11    2.00
12    1.00
Name: rating, dtype: float64
Correlation: -0.33


**Finding:** Average rating drops from about 4.2 at 2-3 days to around 3 at 8-9 days (correlation -0.33, a moderate negative link). Very few orders take 10+ days, so ignore the wobbles at the end. Faster delivery is one of the clearest ways to protect ratings.

## Q4. Is COD riskier? Compare return rate by payment_mode.

In [5]:
(df.groupby("payment_mode")["is_returned"].mean() * 100).round(1).sort_values(ascending=False)

payment_mode
COD            19.1
Unknown        16.7
Net Banking    13.1
Credit Card    12.6
Debit Card     12.3
UPI            12.1
Name: is_returned, dtype: float64

**Finding:** COD has the highest return rate (19.1%). Prepaid modes are around 12-13%. ShopKart could encourage prepaid payment, for example with a small UPI cashback.

## Q5. Which age group spends the most per order? Use pd.cut with bins [17, 25, 35, 45, 60].

In [6]:
df["age_group"] = pd.cut(df["age"], bins=[17, 25, 35, 45, 60], labels=["18-25", "26-35", "36-45", "46-60"])
df.groupby("age_group", observed=True).agg(orders=("order_id", "count"), avg_order=("revenue", "mean")).round(0)

,orders,avg_order
age_group,,
18-25,1411,2820.0
26-35,2494,3033.0
36-45,963,3019.0
46-60,132,1931.0


**Finding:** 26-35 year olds place half of all orders (2,494) and, together with 36-45, spend the most per order (about ₹3,000). The 46-60 group is small (132 orders) and spends about ₹1,931 per order.

## Q6. Do men and women differ in average order value and return rate?

In [7]:
df.groupby("gender").agg(orders=("order_id", "count"), avg_order=("revenue", "mean"),
                         return_rate=("is_returned", "mean")).round(3)

,orders,avg_order,return_rate
gender,,,
Female,2263,3024.288,0.144
Male,2737,2872.008,0.132


**Finding:** Women spend a bit more per order (₹3,024 vs ₹2,872) and return slightly more often (14.4% vs 13.2%). These gaps are small. They are not worth a separate strategy without testing (Module 3).

## Q7. In Pune, is the return problem in every category or just one?

In [8]:
pune = df[df["city"] == "Pune"]
(pune.groupby("category")["is_returned"].mean() * 100).round(1).sort_values(ascending=False)

category
Fashion        38.8
Electronics    33.7
Home           31.7
Beauty         26.5
Books          20.7
Name: is_returned, dtype: float64

**Finding:** Every category in Pune has a return rate between 20.7% (Books) and 38.8% (Fashion), all well above the overall 13.7%. So the problem is the city (delivery, packaging, or partner), not one product.

## Q8. Which weekday gets the most orders?

In [9]:
df["weekday"] = df["order_date"].dt.day_name()
df["weekday"].value_counts()

weekday
Sunday       782
Monday       721
Tuesday      720
Friday       705
Thursday     701
Wednesday    696
Saturday     675
Name: count, dtype: int64

**Finding:** Sunday gets the most orders (782). Every other day is between 675 and 721. Sunday is a little busier, but the gap is small compared with the swing between months (see 6.7), so festivals matter far more than weekdays.